# Train and evaluate on Google Colab

This notebook runs the full experiment on a free Colab GPU:

1. Download the pinned dataset and build the grouped (leakage-reduced) split.
2. Train YOLOv8n twice: on the grouped split (main result) and on the official RF100 split (for comparison).
3. Evaluate both on their own test sets, sweep confidence thresholds on validation data, and draw figures.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Expect roughly an hour per training run; the time per epoch in the training log gives a better estimate. Checkpoints and results are saved to Google Drive in `MyDrive/asbestos_runs`, so a disconnect does not lose finished epochs.

In [ ]:
!nvidia-smi -L  # should list a GPU

## 1. Connect Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/asbestos_runs"  # checkpoints, metadata and results go here
!mkdir -p {RUNS}

## 2. Get the code and install it

In [ ]:
![ -d /content/asbestos-fiber-detection ] || git clone -q https://github.com/sp-ankitprajapati/asbestos-fiber-detection /content/asbestos-fiber-detection
%cd /content/asbestos-fiber-detection
!pip install -q -r requirements.txt

## 3. Download the data and build the grouped split

`prepare_data.py` downloads a fixed revision of the dataset and re-runs the audit.
`make_grouped_split.py` keeps images that share a source name (or identical bytes) in the same split.
The grouping is deterministic (seed 42), so re-running this cell after a disconnect rebuilds the same split.

In [ ]:
!python prepare_data.py
![ -f asbestos/grouped/data.yaml ] || python make_grouped_split.py

## 4. Train on the grouped split (main model)

Fine-tunes COCO-pretrained YOLOv8n for up to 100 epochs. Training stops early if the validation score has not improved for 20 epochs.

In [ ]:
!python train.py --data asbestos/grouped/data.yaml --name grouped --project {RUNS} --output {RUNS}/grouped.pt --device 0 --workers 2

**Only if Colab disconnected during training:** run sections 1–3 again, then remove the `#` from the matching line below and run it to continue from the last saved epoch. If you started a run more than once, use the newest folder (for example `grouped2`).

In [ ]:
# !python train.py --resume {RUNS}/grouped/weights/last.pt --output {RUNS}/grouped.pt --device 0 --workers 2
# !python train.py --resume {RUNS}/official/weights/last.pt --output {RUNS}/official.pt --device 0 --workers 2

## 5. Train on the official RF100 split (for comparison)

Same settings, but on the official split, where variants of the same source image sit in both train and test.
Comparing the two test scores shows how much the official split may overstate performance.

In [ ]:
!python train.py --data asbestos/data.local.yaml --name official --project {RUNS} --output {RUNS}/official.pt --device 0 --workers 2

## 6. Evaluate

- **Validation first:** standard mAP, plus a confidence-threshold sweep for choosing a review setting.
- **Then each model on its own split's test set, once.** The two test sets contain different images, so treat a gap as a warning sign, not an exact measure of leakage.

In [ ]:
!mkdir -p reports/results
!python evaluate.py --weights {RUNS}/grouped.pt --data asbestos/grouped/data.yaml --split val --output reports/results/grouped_val.json --device 0
!python threshold_sweep.py --weights {RUNS}/grouped.pt --data asbestos/grouped/data.yaml --output reports/results/grouped_val_thresholds.csv --device 0
!python evaluate.py --weights {RUNS}/grouped.pt --data asbestos/grouped/data.yaml --split test --output reports/results/grouped_test.json --device 0
!python evaluate.py --weights {RUNS}/official.pt --data asbestos/data.local.yaml --split test --output reports/results/official_test.json --device 0

In [ ]:
import json

for name in ["grouped_val", "grouped_test", "official_test"]:
    result = json.load(open(f"reports/results/{name}.json"))
    print(f"{name:14s} mAP50 {result['map50']:.3f}   mAP50-95 {result['map50_95']:.3f}")
    for row in result["per_class"]:
        print(f"    {row['class']:17s} AP50 {row['map50']:.3f}")

## 7. Figures

`make_figures.py` redraws the README charts and adds the per-class AP comparison.
The sample image shows the grouped model on the first three grouped test images, sorted by file name (not hand-picked), at confidence 0.25.

In [ ]:
!python make_figures.py

In [ ]:
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO(f"{RUNS}/grouped.pt")
paths = sorted(Path("asbestos/grouped/test/images").glob("*.jpg"))[:3]
plots = [r.plot(line_width=2) for r in model.predict([str(p) for p in paths], conf=0.25, verbose=False)]
tiles = [cv2.resize(im, (round(im.shape[1] * 480 / im.shape[0]), 480)) for im in plots]
cv2.imwrite("assets/sample_predictions.jpg", np.hstack(tiles))  # plot() and imwrite both use BGR

## 8. Save the results to Drive

Copies everything worth committing into `MyDrive/asbestos_runs/to_commit`, laid out like the repository.
Add those files to the repo, then fill in the Results table in `README.md`. Checkpoints (`.pt`) stay in Drive.

In [ ]:
!cp {RUNS}/grouped.metadata.json {RUNS}/official.metadata.json reports/results/
!mkdir -p {RUNS}/to_commit/reports
!cp -r reports/results {RUNS}/to_commit/reports/
!cp -r assets {RUNS}/to_commit/
!ls -R {RUNS}/to_commit